#Connection to JDBC SQL Server Injection

In [0]:
server = "asql-salesproject-yash-s1.database.windows.net"
port = "1433"
database = "ASQL_SalesProject_Yash_Ingestion"
username = "sales"
password = "Yash1234"
#JDBC URL
jdbc_url_src = f"jdbc:sqlserver://{server}:{port};databaseName={database}"
#Connection Properties
connection_properties_src = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Connection to JDBC SQL Server Integration

In [0]:
server = "asql-salesproject-yash-s1.database.windows.net"
port = "1433"
database = "ASQL_SalesProject_Yash_Integration"
username = "sales"
password = "Yash1234"
#JDBC URL
jdbc_url_tgt = f"jdbc:sqlserver://{server}:{port};databaseName={database}"
#Connection Properties
connection_properties_tgt = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ============================================================
# SCD TYPE 2 - TBL_DIM_ORDER
# ============================================================

# ============================================================
# 1. READ SOURCE TABLES
# ============================================================

# ORDER_DETAILS
df_order_details = spark.read.jdbc(url=jdbc_url_src,table="dbo.Order_Detail",properties=connection_properties_src)

# ORDER_HEADER
df_order_header = spark.read.jdbc(url=jdbc_url_src,table="dbo.Order_Header",properties=connection_properties_src)

# WAREHOUSE LOOKUP
df_warehouse = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_WAREHOUSE_LKP",properties=connection_properties_tgt)

# TARGET TABLE
df_order_tgt = spark.read.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_ORDER",properties=connection_properties_tgt)

# ============================================================
# 2. PREPARE SOURCE DATA
# ============================================================

# JOIN ORDER_DETAILS WITH ORDER_HEADER
df_src_join = (df_order_details.alias("od").join(df_order_header.alias("oh"),col("od.ORDER_NUMBER") == col("oh.ORDER_NUMBER"),"left"))

# JOIN WITH WAREHOUSE LOOKUP
df_src_final = (df_src_join.alias("src").join(df_warehouse.alias("wh"),col("oh.SALES_BRANCH_CODE") == col("wh.WAREHOUSE_BRANCH_CODE"),"left")
                .select(
        # BUSINESS KEYS
                        col("od.ORDER_DETAIL_CODE"),
                        col("od.ORDER_NUMBER"),

        # SCD2 COLUMNS
                        col("wh.WAREHOUSE_BRANCH_CODE"),
                        col("wh.COUNTRY_CODE"),
                        col("wh.CITY").alias("WAREHOUSE_CITY"),
                        col("od.SHIP_DATE"),
                        col("od.QUANTITY"),
                        col("od.UNIT_COST"),
                        col("od.UNIT_PRICE"),
                        col("od.UNIT_SALE_PRICE"),
                        col("oh.ORDER_CLOSE_DATE"),

        # PASS THROUGH
                        col("oh.ORDER_DATE"),

        # AUDIT COLUMNS
                        lit("INTERNAL DB").alias("SOURCE_ID"),
                        current_timestamp().cast("date").alias("DATA_DATE"),
                        current_timestamp().cast("date").alias("UPDATE_DATE"),
                        current_timestamp().cast("date").alias("BEGIN_EFFECTIVE_DATE"),
                        lit("2099-12-31").cast("date").alias("END_EFFECTIVE_DATE"),
                        lit(1).alias("VERSION_NUM"),
                        lit("Y").alias("CHANGE_INFO")
                        )
                )

# ============================================================
# 3. ACTIVE RECORDS FROM TARGET
# ============================================================

df_active_tgt = df_order_tgt.filter(col("CHANGE_INFO") == "Y")

# ============================================================
# 4. FIND CHANGED RECORDS (SCD2)
# ============================================================

changed_records = (df_src_final.alias("src").join(df_active_tgt.alias("tgt"),
        (
            (col("src.ORDER_DETAIL_CODE") == col("tgt.ORDER_DETAIL_CODE")) &
            (col("src.ORDER_NUMBER") == col("tgt.ORDER_NUMBER"))
        ),"inner")
    .filter(

        (coalesce(col("src.WAREHOUSE_BRANCH_CODE"), lit(-1)) != coalesce(col("tgt.WAREHOUSE_BRANCH_CODE"), lit(-1))) |

        (coalesce(col("src.COUNTRY_CODE"), lit(-1)) != coalesce(col("tgt.COUNTRY_CODE"), lit(-1))) |

        (coalesce(col("src.WAREHOUSE_CITY"), lit("X")) != coalesce(col("tgt.WAREHOUSE_CITY"), lit("X"))) |

        (coalesce(col("src.QUANTITY"), lit(-1)) != coalesce(col("tgt.QUANTITY"), lit(-1))) |

        (coalesce(col("src.UNIT_COST"), lit(-1)) != coalesce(col("tgt.UNIT_COST"), lit(-1))) |

        (coalesce(col("src.UNIT_PRICE"), lit(-1)) != coalesce(col("tgt.UNIT_PRICE"), lit(-1))) |

        (coalesce(col("src.UNIT_SALE_PRICE"), lit(-1)) != coalesce(col("tgt.UNIT_SALE_PRICE"), lit(-1))) |

        (coalesce(col("src.ORDER_CLOSE_DATE"), lit("1900-01-01")) != coalesce(col("tgt.ORDER_CLOSE_DATE"), lit("1900-01-01")))
    )
)

# ============================================================
# 5. EXPIRE OLD RECORDS
# ============================================================

expired_records = (changed_records.select("tgt.*").withColumn("END_EFFECTIVE_DATE",date_sub(current_date(), 1))
                   .withColumn("CHANGE_INFO",lit("N"))
                   .withColumn("UPDATE_DATE",current_date())
                   )

# ============================================================
# 6. CREATE NEW VERSION RECORDS
# ============================================================

new_version_records = (changed_records.select("src.*", col("tgt.VERSION_NUM").alias("OLD_VERSION"))
                       .withColumn("VERSION_NUM",col("OLD_VERSION") + 1).drop("OLD_VERSION")
                       )

# ============================================================
# 7. FIND BRAND NEW RECORDS
# ============================================================

new_records = (df_src_final.alias("src").join(df_active_tgt.alias("tgt"),
        (
            (col("src.ORDER_DETAIL_CODE") == col("tgt.ORDER_DETAIL_CODE")) &
            (col("src.ORDER_NUMBER") == col("tgt.ORDER_NUMBER"))
        ),"left_anti")
               )

# ============================================================
# 8. FINAL INSERT DATA
# ============================================================

final_insert_df = (new_version_records.unionByName(new_records))

# ============================================================
# 9. REMOVE OLD ACTIVE RECORDS FROM TARGET
# ============================================================

unchanged_target = (df_order_tgt.alias("tgt").join(expired_records.select("ORDER_DETAIL_CODE","ORDER_NUMBER").alias("exp"),
        (
            (col("tgt.ORDER_DETAIL_CODE") == col("exp.ORDER_DETAIL_CODE")) &
            (col("tgt.ORDER_NUMBER") == col("exp.ORDER_NUMBER"))
        ),"left_anti")
                    )

# ============================================================
# 10. FINAL TARGET DATA
# ============================================================

final_target_df = (unchanged_target.unionByName(expired_records).unionByName(final_insert_df))

# ============================================================
# 11. GENERATE SURROGATE KEY
# ============================================================

window_spec = Window.orderBy(monotonically_increasing_id())

final_target_df = (final_target_df.withColumn("ORDER_KEY",row_number().over(window_spec)))

# ============================================================
# 12. WRITE FINAL DATA TO TARGET
# ============================================================

final_target_df.write.jdbc(url=jdbc_url_tgt,table="dbo.TBL_DIM_ORDER",mode="overwrite",properties=connection_properties_tgt)

print("SCD TYPE 2 LOAD COMPLETED SUCCESSFULLY")
